# Notebook 02 — Classification Analysis

Behavioural feature exploration + genotype classification.

## Stages
1. Load compact_tracks.csv and map genotypes
2. Extract frame-level behavioural features
3. Aggregate to per-fly features
4. Exploratory visualisation (box plots per genotype, WT vs mutant)
5. Classification (LDA, Logistic, SVC) with cross-validation
6. Feature importance plots

**Replace all `PLACEHOLDER` paths below with your actual paths.**

In [ ]:
import sys
sys.path.insert(0, '..')  # so 'src' is importable from notebooks/

import os
import pandas as pd
import plotly.express as px

from src.classification import (
    map_vial_to_genotype,
    run_classifier,
    plot_by_genotype,
    plot_wt_vs_mutant,
)
from src.features import extract_behavioral_features, aggregate_per_fly_features

## 1 — Configuration

In [ ]:
# ---- EDIT THIS ----
RUN_DIR      = r"../outputs/run_64_41DPE_n004"
FIGURES_DIR  = os.path.join(RUN_DIR, "classification")

os.makedirs(FIGURES_DIR, exist_ok=True)

## 2 — Load data and map genotypes

`map_vial_to_genotype` parses the filename to infer which vial corresponds
to which genotype (e.g. `..._hTDP43_WT-Het-Homo_...`).

In [ ]:
df_raw = map_vial_to_genotype(RUN_DIR)
print(df_raw.shape)
print(df_raw["genotype"].value_counts())
df_raw.head()

## 3 — Extract behavioural features

Computes frame-level kinematics (velocity, acceleration, turning angle),
convex-hull area, and path tortuosity for each fly.

In [ ]:
df_feat = extract_behavioral_features(df_raw)
print(df_feat.shape)
df_feat[["compact_id", "frame", "velocity", "turning_angle", "area_covered", "tortuosity"]].head()

## 4 — Aggregate to per-fly features

In [ ]:
df_agg = aggregate_per_fly_features(df_feat, pause_threshold=1.0)

# Merge genotype label
genotype_map = df_raw.drop_duplicates("compact_id").set_index("compact_id")["genotype"]
df_agg["genotype"] = df_agg["compact_id"].map(genotype_map)
df_agg = df_agg.dropna(subset=["genotype"])

print(df_agg.shape)
df_agg.head()

## 5 — Exploratory visualisation

Box plots for each feature, grouped by genotype.

In [ ]:
FEATURES = [
    "mean_velocity",
    "median_velocity",
    "pause_fraction",
    "total_distance_traveled",
    "tortuosity",
    "area_covered",
]

FEATURE_TITLES = {
    "mean_velocity":            "Mean velocity (px/s)",
    "median_velocity":          "Median velocity (px/s)",
    "pause_fraction":           "Pause fraction",
    "total_distance_traveled":  "Total distance traveled (px)",
    "tortuosity":               "Path tortuosity",
    "area_covered":             "Area covered (px²)",
}

hover_data = ["compact_id"]

plot_by_genotype(df_agg, FEATURES, FEATURE_TITLES, hover_data, outdir=FIGURES_DIR)

In [ ]:
plot_wt_vs_mutant(df_agg, FEATURES, FEATURE_TITLES, hover_data, outdir=FIGURES_DIR)

## 6 — Classification

Train LDA, Logistic Regression, and SVC classifiers.
Cross-validation accuracy and feature importance figures are saved to `FIGURES_DIR`.

In [ ]:
for model_name in ["lda", "logistic", "svc"]:
    for mode in ["multiclass", "binary"]:
        print(f"\n=== {model_name.upper()} [{mode}] ===")
        run_classifier(
            df=df_agg,
            outdir=FIGURES_DIR,
            model_name=model_name,
            classification_mode=mode,
            cv=5,
            plot_importance=True,
        )

## Summary

All figures (HTML + PNG) have been saved to `FIGURES_DIR`.
Open the HTML files in a browser for interactive Plotly charts.